# 03 - Fee Analysis And Claim Consistency Checks

This notebook computes fee run-rates and compares observed fee productivity against the
widely cited ~0.66% annual fee framing.

**Source class:** API_DERIVED for fees/TVL, then deterministic arithmetic checks.


## Analysis Steps

1. Pull latest DeFiLlama snapshots (optional, enabled by default).
2. Compute 30d / 90d fee run-rates and implied annualized fee rate.
3. Run consistency checks from `src/reconciliation/checks.py`.
4. Visualize fee dispersion and trend.


In [ ]:
from pathlib import Path
import sys
import json
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

sns.set_theme(style="whitegrid")

cwd = Path.cwd().resolve()
root = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / "notebooks" / "notebook_utils.py").exists():
        root = candidate
        break
if root is None:
    raise RuntimeError("Could not locate repository root containing notebooks/notebook_utils.py")

if root.as_posix() not in sys.path:
    sys.path.insert(0, root.as_posix())

from notebooks import notebook_utils as nbx
ROOT = nbx.setup_notebook_context()
print(f"Project root: {ROOT}")
print(f"Notebook run started (UTC): {nbx.utc_now_iso()}")


In [ ]:
from src.reconciliation.checks import check_fee_rate_consistency

REFRESH_DEFILLAMA = True
if REFRESH_DEFILLAMA:
    nbx.refresh_defillama_snapshots()

lazy_tvl_path = nbx.latest_defillama_snapshot("lazy-summer-protocol", "tvl")
fees_path = nbx.latest_defillama_snapshot("lazy-summer-protocol", "fees")

lazy_tvl = nbx.protocol_tvl_df(nbx.load_json(lazy_tvl_path))
fees_df = nbx.fee_series_df(nbx.load_json(fees_path))

if lazy_tvl.empty or fees_df.empty:
    raise RuntimeError("Missing TVL or fee data after snapshot load.")


In [ ]:
metrics_30 = nbx.summarize_fee_productivity(lazy_tvl, fees_df, fee_window_days=30)
metrics_90 = nbx.summarize_fee_productivity(lazy_tvl, fees_df, fee_window_days=90)

observed_rate = metrics_90.get("implied_fee_rate") if metrics_90.get("status") == "OK" else None
consistency = check_fee_rate_consistency(
    computed_rate=float(observed_rate or 0.0),
    claimed_rate=0.0066,
    tolerance=0.002,
)

summary = pd.DataFrame([
    {
        "window": "30d",
        "fees_annualized_usd": metrics_30.get("fees_annualized_usd"),
        "avg_tvl_usd": metrics_30.get("avg_tvl_usd"),
        "implied_fee_rate": metrics_30.get("implied_fee_rate"),
    },
    {
        "window": "90d",
        "fees_annualized_usd": metrics_90.get("fees_annualized_usd"),
        "avg_tvl_usd": metrics_90.get("avg_tvl_usd"),
        "implied_fee_rate": metrics_90.get("implied_fee_rate"),
    },
])

display(summary)
display(pd.DataFrame([consistency]))


In [ ]:
fees_plot = fees_df.copy()
fees_plot["fees_30d_ma"] = fees_plot["dailyFeesUSD"].rolling(30, min_periods=7).mean()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)

axes[0].plot(fees_plot["date"], fees_plot["dailyFeesUSD"], color="#9ecae1", linewidth=1.0, label="Daily fees")
axes[0].plot(fees_plot["date"], fees_plot["fees_30d_ma"], color="#08519c", linewidth=2.0, label="30d moving average")
axes[0].set_title("Daily Fee Trend")
axes[0].set_ylabel("USD")
axes[0].legend()

axes[1].hist(fees_plot["dailyFeesUSD"], bins=30, color="#74c476", alpha=0.8)
axes[1].axvline(fees_plot["dailyFeesUSD"].mean(), color="#00441b", linestyle="--", linewidth=1.6, label="Mean")
axes[1].set_title("Distribution Of Daily Fees")
axes[1].set_xlabel("USD/day")
axes[1].legend()

plt.show()


In [ ]:
verdict = "PASS" if consistency["passes"] else "PARTIAL"
message = (
    f"Observed 90d implied fee rate = {nbx.fmt_pct(observed_rate)}; "
    f"claimed baseline = {nbx.fmt_pct(consistency['claimed_rate'])}; "
    f"abs diff = {nbx.fmt_pct(consistency['diff'])}."
)

display(Markdown(f"**Consistency verdict:** `{verdict}`  \n{message}"))
